In [2]:
from pymatgen.core import Structure
from pymatgen.core.periodic_table import Element
import numpy as np

spinel_structure = Structure.from_file("/home/gaotianjiao/modnet-master/ext_struc/spinels_cif/mp-14246.cif")
print(f"晶胞中总原子数: {len(spinel_structure)}")
print(f"化学式: {spinel_structure.formula}")
print(f"晶格参数: a={spinel_structure.lattice.a:.4f}, "
      
      f"b={spinel_structure.lattice.b:.4f}, "
      f"c={spinel_structure.lattice.c:.4f}")
print(f"晶格角度: alpha={spinel_structure.lattice.alpha:.2f}, "
      f"beta={spinel_structure.lattice.beta:.2f}, "
      f"gamma={spinel_structure.lattice.gamma:.2f}")

晶胞中总原子数: 84
化学式: Ba12 Al24 S48
晶格参数: a=12.7321, b=12.7321, c=12.7321
晶格角度: alpha=90.00, beta=90.00, gamma=90.00


/home/gaotianjiao/anaconda3/envs/python3.10/lib/python3.10/site-packages/pymatgen/core/structure.py:3175: EncodingWarning: We strongly encourage explicit `encoding`, and we would use UTF-8 by default as per PEP 686
  with zopen(filename, mode="rt", errors="replace") as file:


In [3]:
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
def identify_asymmetric_unit(structure, symprec=0.01):
    """识别不对称单元中的不等价原子
    
    Args:
        structure: pymatgen Structure 对象
        symprec: 对称性分析精度
    
    Returns:
        tuple: (inequivalent_sites, equivalent_sites, space_group_info ,asymmetric_structure)
    """
    analyzer = SpacegroupAnalyzer(structure, symprec=symprec)
    
    # 获取空间群信息
    space_group = analyzer.get_space_group_number()
    space_group_symbol = analyzer.get_space_group_symbol()
    
    # 获取对称化结构
    asymmetric_structure = analyzer.get_symmetrized_structure()
    
    # 获取不等价原子位点-对于每组等价原子，取第一个原子作为代表
    equivalent_sites = asymmetric_structure.equivalent_sites
    inequivalent_sites = [sites[0] for sites in equivalent_sites]
    
    return inequivalent_sites, equivalent_sites, (space_group, space_group_symbol),asymmetric_structure

In [4]:
inequivalent_sites, equivalent_sites, space_group_info,symmetrized_structure = identify_asymmetric_unit(spinel_structure)

In [5]:
def calculate_all_angles(center, neighbor_coords):
    """
    计算中心原子到所有邻居的两两夹角，返回 n x n 上三角矩阵
    """
    vectors = neighbor_coords - center  # shape: (n,3)
    norms = np.linalg.norm(vectors, axis=1)
    n = len(neighbor_coords)
    angles = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            dot = np.dot(vectors[i], vectors[j])
            angles[i, j] = np.degrees(np.arccos(np.clip(dot / (norms[i]*norms[j]), -1.0, 1.0)))
    return angles

In [6]:
def classify_coordination(site, neighbors, angle_tolerance=15):
    """
    根据邻居几何手动分类配位环境，同时判断是否扭曲
    - CN=2~12
    - 返回字典：{'environment': str, 'distorted': bool}
    """
    n = len(neighbors)
    if n == 0:
        return {'environment': "isolated", 'distorted': False}

    center = np.array(site.coords)
    neighbor_coords = np.array([n['coords'] for n in neighbors])
    angles = calculate_all_angles(center, neighbor_coords)
    angles_flat = angles[np.triu_indices(n, k=1)]
    angles_flat = np.round(angles_flat, 2)

    # -----------------------------
    # 初始化
    # -----------------------------
    environment = "unknown"
    distorted = False

    # -----------------------------
    # 按 CN 分类
    # -----------------------------
    if n == 2:
        if abs(angles_flat[0]-180)<angle_tolerance:
            environment = "linear"
            distorted = False
        else:
            environment = "bent"
            distorted = True

    elif n == 3:
        if np.all(np.abs(angles_flat-120)<angle_tolerance):
            environment = "trigonal_planar"
            distorted = False
        else:
            environment = "trigonal_pyramidal"
            distorted = True

    elif n == 4:
        ninety_count = np.sum(np.abs(angles_flat - 90) < angle_tolerance)
        one_eighty_count = np.sum(np.abs(angles_flat - 180) < angle_tolerance)
        one_o_nine = np.sum(np.abs(angles_flat - 109.47) < angle_tolerance)

        if one_o_nine == 6:
            environment = "tetrahedral"
            distorted = np.any(np.abs(angles_flat - 109.47) > angle_tolerance)
        elif ninety_count >= 4 and one_eighty_count >= 2:
            environment = "square_planar"
            distorted = not (ninety_count >= 4 and one_eighty_count >= 2)
        else:
            environment = "4-coordinated"
            distorted = True

    elif n == 5:
        ninety_count = np.sum(np.abs(angles_flat - 90) < angle_tolerance)
        oneeighty_count = np.sum(np.abs(angles_flat - 180) < angle_tolerance)
        one_twenty_count = np.sum(np.abs(angles_flat - 120) < angle_tolerance)

        if one_twenty_count >= 3 and oneeighty_count >= 1 and ninety_count >= 3:
            environment = "trigonal_bipyramidal"
            distorted = not (one_twenty_count >= 3 and oneeighty_count >= 1 and ninety_count >= 3)
        else:
            environment = "square_pyramidal"
            distorted = True

    elif n == 6:
        ninety_count = np.sum(np.abs(angles_flat-90)<angle_tolerance)
        oneeighty_count = np.sum(np.abs(angles_flat-180)<angle_tolerance)
        if ninety_count>=8 and oneeighty_count>=3:
            environment = "octahedral"
            distorted = not (ninety_count>=8 and oneeighty_count>=3)
        else:
            environment = "6-coordinated"
            distorted = True

    elif n == 8:
        environment = "cubic"
        distorted = False
    elif n == 12:
        environment = "cuboctahedral"
        distorted = False
    else:
        environment = f"{n}-coordinated"
        distorted = False

    return {'environment': environment, 'distorted': distorted}

In [7]:
from pymatgen.analysis.local_env import CrystalNN
import numpy as np

def get_crystalnn_motifs(structure, symmetrized_structure, weight_threshold=0.8, angle_tolerance=15):
    """
    使用 CrystalNN 生成结构基元（每个原子都生成），标记等价组和代表原子。
    
    Args:
        structure: pymatgen Structure 对象
        symmetrized_structure: pymatgen SymmetrizedStructure 对象
        weight_threshold: CrystalNN 邻居权重阈值
        angle_tolerance: 角度分类扭曲阈值

    Returns:
        motifs: list of dict，每个 dict 为一个基元
    """
    cnn = CrystalNN()
    motifs = []
    global_indices = set()  # 检查是否覆盖所有原子

    for group_idx, group in enumerate(symmetrized_structure.equivalent_sites):
        wyckoff_symbol = symmetrized_structure.wyckoff_symbols[group_idx]
        representative_index = structure.sites.index(group[0])  # 每组的代表原子

        for site in group:
            site_index = structure.sites.index(site)
            global_indices.add(site_index)

            # 获取邻居信息
            nn_info = cnn.get_nn_info(structure, site_index)
            nn_info = [n for n in nn_info if n['weight'] >= weight_threshold]

            neighbors = []
            for n in nn_info:
                vector = n['site'].coords - site.coords
                neighbors.append({
                    'species': n['site'].species_string,
                    'coords': n['site'].coords,
                    'weight': n['weight'],
                    'distance': np.linalg.norm(vector),
                    'vector': vector,
                    'index': n['site_index']  # 全局索引
                })

            # 判断配位环境及扭曲
            coord_info = classify_coordination(site, neighbors, angle_tolerance)

            motif = {
                'central_atom': {
                    'species': site.species_string,
                    'coords': site.frac_coords,
                    'wyckoff': wyckoff_symbol,
                    'index': site_index,
                    'equivalent_group_index': group_idx,
                    'representative_index': representative_index
                },
                'coordination_number': len(neighbors),
                'coordination_environment': coord_info['environment'],
                'distorted': coord_info['distorted'],
                'neighbors': neighbors
            }
            motifs.append(motif)

    # 检查是否覆盖所有原子
    all_indices = set(range(len(structure.sites)))
    missing_indices = all_indices - global_indices
    if missing_indices:
        print(f"Warning: the following atoms were not included as centers: {missing_indices}")

    return motifs


In [8]:
# 生成基元
structural_motifs = get_crystalnn_motifs(
    spinel_structure,
    symmetrized_structure
)

# 只提取等价组代表原子的基元做局部环境特征
representative_motifs = [
    m for m in structural_motifs
    if m['central_atom']['index'] == m['central_atom']['representative_index']
]

print(f"Total motifs: {len(structural_motifs)}")
print(f"Representative motifs: {len(representative_motifs)}")


/home/gaotianjiao/anaconda3/envs/python3.10/lib/python3.10/site-packages/pymatgen/analysis/local_env.py:4232: UserWarning: No oxidation states specified on sites! For better results, set the site oxidation states in the structure.
  warnings.warn(
/home/gaotianjiao/anaconda3/envs/python3.10/lib/python3.10/site-packages/pymatgen/analysis/local_env.py:4025: UserWarning: CrystalNN: cannot locate an appropriate radius, covalent or atomic radii will be used, this can lead to non-optimal results.
  warnings.warn(


Total motifs: 84
Representative motifs: 5


In [9]:
structural_motifs[0]

{'central_atom': {'species': 'Ba',
  'coords': array([0.37299784, 0.12700216, 0.87299784]),
  'wyckoff': '8c',
  'index': 0,
  'equivalent_group_index': 0,
  'representative_index': 0},
 'coordination_number': 6,
 'coordination_environment': '6-coordinated',
 'distorted': True,
 'neighbors': [{'species': 'S',
   'coords': array([ 3.4637393 ,  0.31949566, 13.75541042]),
   'weight': 1,
   'distance': 3.210441244513378,
   'vector': array([-1.28529185, -1.29750356,  2.64034891]),
   'index': 46},
  {'species': 'S',
   'coords': array([ 7.38938006,  2.90229107, 12.41256507]),
   'weight': 1,
   'distance': 3.2104412445133783,
   'vector': array([2.64034891, 1.28529185, 1.29750356]),
   'index': 43},
  {'species': 'S',
   'coords': array([ 6.04653471, -1.02334969,  9.82976966]),
   'weight': 1,
   'distance': 3.210441244513379,
   'vector': array([ 1.29750356, -2.64034891, -1.28529185]),
   'index': 39},
  {'species': 'S',
   'coords': array([ 3.69548712,  4.7338474 , 10.94220737]),
   'we

In [9]:
import sys
sys.path.append("/home/gaotianjiao/modnet-master/ext_struc")

In [10]:
from CE_feature import extract_CE_features_from_motifs
ce_features, property_names = extract_CE_features_from_motifs(structural_motifs, "/home/gaotianjiao/modnet-master/ext_struc/element_properties/ElementsProperties.xlsx")

In [11]:
from CE_feature import weighted_CE_vector
final_CE_vector = weighted_CE_vector(ce_features, method='hybrid')
print(final_CE_vector)

C_0       24.383378
C_1       54.860885
C_2     1519.000485
C_3        0.302846
C_4     2583.214567
           ...     
E_32     122.913950
E_33       1.443781
E_34       1.587315
E_35       4.358315
E_36       2.618997
Length: 74, dtype: float64


In [12]:
import pandas as pd
ce_vectors = [f['CE_feature'] for f in ce_features]
ce_df = pd.DataFrame(ce_vectors)

In [13]:
from CE_feature import rename_CE_feature_columns
ce_df = rename_CE_feature_columns(ce_df, property_names)
print(ce_df.head(15))

    C_Atomic number start counting left top, left-right sequence  \
0                                                56.0              
1                                                56.0              
2                                                56.0              
3                                                56.0              
4                                                56.0              
5                                                56.0              
6                                                56.0              
7                                                56.0              
8                                                56.0              
9                                                56.0              
10                                               56.0              
11                                               56.0              
12                                               13.0              
13                                              

In [14]:
print(type(ce_df))

<class 'pandas.core.frame.DataFrame'>
